In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from utils import ChocolateCountDataset, ChocolateCounterNetDeep
from tqdm import tqdm
import os
import pandas as pd

from sklearn.model_selection import train_test_split


In [2]:
import os

all_filenames = sorted([f for f in os.listdir("../dataset_project_iapr2025/train") if f.endswith(('.JPG'))])

train_filenames, val_filenames = train_test_split(all_filenames[13:], test_size=0.33, random_state=42)

train_filenames += all_filenames[:13]

In [5]:

# Paths (adjust as needed)
TRAIN_IMAGE_DIR = "../dataset_project_iapr2025/train"
VAL_IMAGE_DIR = "../dataset_project_iapr2025/train"
LABEL_DIR = "./labels"

# Hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 400
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PATH = "best_model.pth"

# Dataset and loaders
train_dataset = ChocolateCountDataset(
    image_filenames=train_filenames,
    image_dir=TRAIN_IMAGE_DIR,
    label_dir=LABEL_DIR,
    augment=True,
    n_augmentations=4
)
val_dataset = ChocolateCountDataset(
    image_filenames=val_filenames,
    image_dir=VAL_IMAGE_DIR,
    label_dir=LABEL_DIR,
    augment=False
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Model, loss, optimizer
model = ChocolateCounterNetDeep().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# Training loop
def evaluate(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)

best_val_loss = float("inf")

print("Start training...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0
    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images, targets = images.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    avg_train_loss = running_loss / len(train_loader.dataset)
    val_loss = evaluate(val_loader)

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"Model saved to {SAVE_PATH}")

torch.save(model.state_dict(), "last_model.pth")
print("Model saved to last_model.pth")

Start training...


Epoch 1/400: 100%|██████████| 12/12 [00:03<00:00,  3.83it/s]


Epoch 1: Train Loss = 0.6893, Val Loss = 0.8144
Model saved to best_model.pth


Epoch 2/400: 100%|██████████| 12/12 [00:03<00:00,  3.85it/s]


Epoch 2: Train Loss = 0.6041, Val Loss = 0.8575


Epoch 3/400: 100%|██████████| 12/12 [00:03<00:00,  3.87it/s]


... (657 outputs left out)

Epoch 332/400: 100%|██████████| 12/12 [00:03<00:00,  3.12it/s]


Epoch 332: Train Loss = 0.0346, Val Loss = 0.6877


Epoch 333/400: 100%|██████████| 12/12 [00:03<00:00,  3.09it/s]


KeyboardInterrupt: 

In [4]:
TEST_IMAGE_DIR = "../dataset_project_iapr2025/test"
MODEL_CHECKPOINT = "best_model.pth"
OUTPUT_CSV_PATH = "submission.csv"
CLASS_NAMES_CSV = [
    "Jelly White", "Jelly Milk", "Jelly Black", "Amandina", "Crème brulée",
    "Triangolo", "Tentation noir", "Comtesse", "Noblesse", "Noir authentique",
    "Passion au lait", "Arabia", "Stracciatella"
]

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ChocolateCounterNetDeep().to(device)
model.load_state_dict(torch.load(MODEL_CHECKPOINT, map_location=device))
model.eval()

# Test dataset
test_filenames = sorted([f for f in os.listdir("../dataset_project_iapr2025/test") if f.endswith(('.JPG'))])
test_dataset = ChocolateCountDataset(image_filenames=test_filenames, image_dir=TEST_IMAGE_DIR, label_dir=None, augment=False)
test_dataset.samples = test_filenames  # Manually set samples
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Predict and collect results
results = []

with torch.no_grad():
    for fname, (image, _) in zip(test_filenames, test_loader):
        image = image.to(device)
        output = model(image).cpu().numpy()[0]
        output = output.round().astype(int).clip(min=0)  # Round to int and avoid negatives

        image_id = os.path.splitext(fname)[0].replace("IMG_", "").replace("L", "")
        row = [image_id] + output.tolist()
        results.append(row)

# Save to CSV
df = pd.DataFrame(results, columns=["id"] + CLASS_NAMES_CSV)
df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Submission file saved to {OUTPUT_CSV_PATH}")

/tmp/ipykernel_142478/1840409507.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_CHECKPOINT, map_location=device))


Submission file saved to submission.csv
